# Compiled Semantic Fashion Search

Interactive held-out fashion search using the same semantic-predicate methods and accounting as the paper.

The main comparison is **Binary1-LS2-int4** (56 B/item, 216 B/predicate) versus **PQ64 compiled linear** (64 B/item, ~65.5 KB/predicate), with dense MiniLM, FP32 linear, and RSA2 baselines. The app also shows the paper's joint-memory accounting and fair live-HNSW latency table.

Try **`minimalist black office shoes not sporty`**. Exact catalog constraints remain exact; latent terms become reusable learned soft predicates. Product images are from the strict held-out split.


In [ ]:
#@title 1) Clone repo and install pinned project extras
import os, pathlib, shutil, subprocess, sys
ROOT = pathlib.Path('/content/ras')
if ROOT.exists():
    shutil.rmtree(ROOT)
subprocess.run(['git','clone','--depth=1','https://github.com/hanialshater/ras.git',str(ROOT)], check=True)
os.chdir(ROOT)
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.[demo,benchmark]'], check=True)

# Colab may contain an unrelated PyPI package named `rsa`; force this repo.
SRC = str(ROOT / 'src')
if SRC in sys.path:
    sys.path.remove(SRC)
sys.path.insert(0, SRC)
for name in list(sys.modules):
    if name == 'ras' or name.startswith('ras.'):
        del sys.modules[name]
import ras
commit = subprocess.check_output(['git','rev-parse','HEAD']).decode().strip()
print('commit:', commit)
print('using ras package:', ras.__file__)
assert str(ROOT / 'src' / 'ras') in str(pathlib.Path(ras.__file__).resolve()), 'Wrong rsa package imported'


In [ ]:
#@title 2) Check runtime
import torch
print('torch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
if not torch.cuda.is_available():
    print('Tip: Runtime → Change runtime type → GPU. CPU works, but first-time CLIP preparation is slower.')


In [ ]:
#@title 3) Prepare strict held-out catalog + semantic programs
# Smoke config keeps the public demo quick. The same code path supports the full config.
import os, time, sys
os.chdir('/content/ras')
SRC = '/content/ras/src'
if SRC in sys.path:
    sys.path.remove(SRC)
sys.path.insert(0, SRC)
from demos.fashion_app import prepare_demo
t0 = time.time()
state = prepare_demo('configs/binary_bbq_smoke.yaml')
print(f'ready in {(time.time()-t0)/60:.1f} minutes')
print(f'visible held-out products: {len(state.df_test):,}')


In [ ]:
#@title 4) Launch the app
from demos.fashion_app import build_app
app = build_app(state)
app.launch(share=True, debug=False)
